# Init Lakehouse

In [1]:
%%configure -f
{
    "defaultLakehouse": {"name": "DE_LH_100_BondedWarehouse"}
}

StatementMeta(, b27fd58b-8e2b-44af-99a1-e8f65a9073b8, -1, Finished, Available, Finished)

# Init Imports (these need cutting-down post creation)

In [2]:
import os
import csv
import re
import shutil
import unicodedata
import pandas as pd

import notebookutils

#from decimal import Decimal
from datetime import datetime
from datetime import timedelta
#from collections import Counter
#from functools import reduce
import time

#from pyspark import StorageLevel
from pyspark.sql import DataFrame, Row
from pyspark.sql.functions import col, lit, when, concat, concat_ws, coalesce, count, monotonically_increasing_id, sum, to_date, udf, current_timestamp, length, substring, split, size, asc, row_number, desc, trim, regexp_replace
from pyspark.sql.functions import broadcast, hash, array, expr, array_distinct, date_format
from pyspark.sql.types import *
#from pyspark.sql import Window
from pyspark.sql import functions as F
from delta.tables import DeltaTable

StatementMeta(, b27fd58b-8e2b-44af-99a1-e8f65a9073b8, 3, Finished, Available, Finished)

# Init Export Process

In [3]:
def save_dataframe_to_csv(df, file_path, show_header=False, mode='overwrite'):
    """
    Save a DataFrame as a single CSV file in a PySpark application.

    Parameters:
    df (pyspark.sql.DataFrame): The DataFrame to save.
    file_path (str): The path to save the CSV file.
    header (bool): Whether to include the header in the CSV file. Default is True.
    mode (str): The write mode. Options are 'overwrite', 'append', 'ignore', 'error' or 'errorifexists'. Default is 'overwrite'.

    Returns:
    None
    """

    use_pipes = len(df.columns) != 1
    print(f'Add pipes: {use_pipes}')
    print(f'Show headers: {show_header}')

    pandas_df = df.toPandas()
    
    # Replace newlines and carriage returns
    pandas_df = pandas_df.replace({r'\r\n': ' ', r'\n': ' ', r'\r': ' '}, regex=True)

    # Create a string representation of the DataFrame with '|' as separator
    # Escape special characters such as commas and pipes
    if use_pipes:
        csv_data = pandas_df.to_csv(sep="|", index=False, header=show_header, quoting=csv.QUOTE_NONE, escapechar="\\")
    else:
        csv_data = pandas_df.to_csv(sep="~", index=False, header=show_header, quoting=csv.QUOTE_NONE, escapechar="\\")

    # Add trailing pipe '|' at the end of each line
    csv_data_with_pipe = '\n'.join([line + '|' for line in csv_data.split('\n') if line])

    # Write to the file
    with open(file_path, 'w') as f:
        f.write(csv_data_with_pipe)


StatementMeta(, b27fd58b-8e2b-44af-99a1-e8f65a9073b8, 4, Finished, Available, Finished)

# Init Debug & Incremental Vars

In [4]:
workspace_name = notebookutils.mssparkutils.env.getWorkspaceName()

if "DEV" in workspace_name.upper():
    debug = True
    incremental_run = False
    default_days_lag: int = 7

elif "UAT" in workspace_name.upper():
    debug = True
    incremental_run = True
    default_days_lag: int = 7

else:
    debug = False
    incremental_run = True
    default_days_lag: int = 1

if debug:
    print(debug , incremental_run)

StatementMeta(, b27fd58b-8e2b-44af-99a1-e8f65a9073b8, 5, Finished, Available, Finished)

True False


# Init Days Lag Var

In [5]:
filterdate_pipe = ''

#default_days_lag: int = 1

enable_string_truncation = True
create_hash_cols: bool = False
transfer_file: bool = False
retain_error_records_in_ouput_file: bool = False

# Override Debug

#debug = False   #<<<<<<<<<<<<<<<<<<<<<<<<<<<  <<<<<<<<<<<<<<<<<<<<
#debug = True   #<<<<<<<<<<<<<<<<<<<<<<<<<<<  <<<<<<<<<<<<<<<<<<<<

# Override Full Run 

#incremental_run = False    #<<<<<<<<<<<<<<<<<<<<<<<<<<<  <<<<<<<<<<<<<<<<<<<<
#incremental_run = True     #<<<<<<<<<<<<<<<<<<<<<<<<<<<  <<<<<<<<<<<<<<<<<<<<

StatementMeta(, b27fd58b-8e2b-44af-99a1-e8f65a9073b8, 6, Finished, Available, Finished)

In [6]:
filterdate = datetime.now() - timedelta(days=default_days_lag)
filterdate = filterdate.date()

if debug:
    print(f'Get Products from: {filterdate}')

StatementMeta(, b27fd58b-8e2b-44af-99a1-e8f65a9073b8, 7, Finished, Available, Finished)

Get Products from: 2025-05-05


# Init Query(s)

In [7]:
products_df = spark.sql(f"""
SELECT 

'ENDCTG' AS Company_Code,
it.itemid AS Product_Code,
c.code AS Commodity_Code,

concat_ws('',
      substr(concat_ws('', pt.name, it.endfabriccomposition), 1, 119)
      ,
      CASE WHEN it.endmenswear = 1 and it.endwomenswear = 1 THEN 'U'  
            WHEN it.endmenswear = 1 THEN 'M' 
            WHEN it.endwomenswear = 1 THEN 'F' 
      ELSE '' END)
       as Description,

v.LangdonCode AS VAT_Rate_Identifier,
CAST(it.netweight AS Decimal(10, 4)) AS Unit_Weight,
'' AS Unit_Cost,
'' AS Unit_Cost_Currency,
'' AS Effective_Date,
'' AS Default_Import_Project,
'' AS Default_CLST_Project,
'' AS Default_IPR_Project,
CASE WHEN ci.additionalunits NOT IN (30,31) OR ci.additionalunits IS NULL THEN 30 ELSE ci.additionalunits END AS SKU,
'' AS BTI,
'' AS EC_Supplementary_Codes,
'' AS IP_Flag

FROM ecoresproduct p 

INNER JOIN inventtable it 
ON it.product = p.recid 

LEFT JOIN ecorescategory c 
ON c.recid = it.intrastatcommodity

LEFT JOIN ecoresproducttranslation pt 
ON pt.product = p.recid 

LEFT JOIN inventtablemodule itm 
ON it.itemid = itm.itemid 
AND it.dataareaid = itm.dataareaid
AND itm.moduletype = 1

LEFT JOIN vatgrouplookup v 
ON v.ItemVATgroup = itm.taxitemgroupid 

LEFT JOIN ecorescategoryintrastat ci 
ON ci.category = it.intrastatcommodity

WHERE it.dataareaid IN ('end.','END.')
AND GREATEST(
  p.modifieddatetime,
  c.modifieddatetime,
  it.modifieddatetime,
  itm.modifieddatetime,
  pt.modifieddatetime,
  ci.SinkModifiedOn
) >= '{filterdate}'

"""
)
if debug:
      display(products_df)

StatementMeta(, b27fd58b-8e2b-44af-99a1-e8f65a9073b8, 8, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, cac471da-3afc-4377-9e62-133eeef1d6d1)

In [8]:
products_df = products_df.select(
    substring(col("Company_Code").cast("string"),1, 10).alias("Company_Code"),
    substring(col("Product_Code").cast("string"),1, 25).alias("Product_Code"),
    substring(col("Commodity_Code").cast("string"),1, 10).alias("Commodity_Code"),
    substring(col("Description").cast("string"),1, 120).alias("Description"),
    substring(col("VAT_Rate_Identifier").cast("string"),1, 1).alias("VAT_Rate_Identifier"),
    substring(col("Unit_Weight").cast("string"),1, 11).alias("Unit_Weight"),
    col("Unit_Cost").cast("string").alias("Unit_Cost"),
    col("Unit_Cost_Currency").cast("string").alias("Unit_Cost_Currency"),
    col("Effective_Date").cast("string").alias("Effective_Date"),
    col("Default_Import_Project").cast("string").alias("Default_Import_Project"),
    col("Default_CLST_Project").cast("string").alias("Default_CLST_Project"),
    col("Default_IPR_Project").cast("string").alias("Default_IPR_Project"),
    substring(col("SKU").cast("string"),1,3).alias("SKU"),
    col("BTI").cast("string").alias("BTI"),
    col("EC_Supplementary_Codes").cast("string").alias("EC_Supplementary_Codes"),
    col("IP_Flag").cast("string").alias("IP_Flag")
)
if debug:
    display(products_df)

#display(products_df.select("Description").withColumn("Description_Length", length(col("Description"))).orderBy(desc("Description_Length")))

StatementMeta(, b27fd58b-8e2b-44af-99a1-e8f65a9073b8, 9, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 76f2585a-9a70-4b5a-93bf-063169039f50)

# Regex Pass to Remove Non-ASCII

In [9]:
products_df = products_df.withColumn("Description", trim(regexp_replace(regexp_replace("Description", "[^\\x00-\\x7F]|[\\|:#/?,Â%(),.-]", ""), "\\s+", " ")))

if debug:
    display(products_df)

StatementMeta(, b27fd58b-8e2b-44af-99a1-e8f65a9073b8, 10, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, a47b1581-111b-4065-9626-ca97ab2138a6)

## Date Field Changing Post Query(s)

In [10]:
list_date_columns_1 = [name for name, dtype in products_df.dtypes if dtype in ('date','timestamp')]

if debug:
    print("Date Columns to change: " , list_date_columns_1)

StatementMeta(, b27fd58b-8e2b-44af-99a1-e8f65a9073b8, 11, Finished, Available, Finished)

Date Columns to change:  []


In [11]:
for column in list_date_columns_1:
    products_df = products_df.withColumn(column, date_format(column, "dd-MM-yyyy"))

StatementMeta(, b27fd58b-8e2b-44af-99a1-e8f65a9073b8, 12, Finished, Available, Finished)

In [12]:
if debug:
    display(products_df)

StatementMeta(, b27fd58b-8e2b-44af-99a1-e8f65a9073b8, 13, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 73fcc410-76fb-4db3-ad8e-bf3489e3640e)

# Init Error Check Process

In [13]:
products_Mandatory_Columns = [
    "Company_Code",
    "Product_Code",
    "Commodity_Code",
    "VAT_Rate_Identifier",
    "SKU"
]

StatementMeta(, b27fd58b-8e2b-44af-99a1-e8f65a9073b8, 14, Finished, Available, Finished)

In [14]:
level = 1

null_condition = None
null_column_names_exprs = []

for column in products_Mandatory_Columns:
    condition = col(column).isNull()
    null_condition = condition if null_condition is None else null_condition | condition
    null_column_names_exprs.append(when(condition, lit(column)))

# Collect error columns into arrays
products_with_errors = products_df.withColumn("null_failed_columns", array(*null_column_names_exprs))

# Filter out nulls from those arrays
products_with_errors = products_with_errors.withColumn(
    "null_failed_columns", expr("filter(null_failed_columns, x -> x is not null)"))

# Generate the error messages (only when columns exist)
products_with_errors = products_with_errors.withColumn(
    "null_errors",
    when(size(col("null_failed_columns")) > 0,
         concat_ws("", lit("Columns "), concat_ws(" , ", col("null_failed_columns")), lit(f" are null at level {level}"))))

# Combine all errors
products_with_errors = products_with_errors.withColumn(
    "error_fields",
    concat_ws(" , ", col("null_errors"))
)

# Filter bad and good
products_bad_df = products_with_errors.filter(null_condition) \
    .drop("null_errors", "null_failed_columns")

products_df = products_with_errors.filter(~(null_condition)) \
    .drop("error_fields", "null_errors", "null_failed_columns")

StatementMeta(, b27fd58b-8e2b-44af-99a1-e8f65a9073b8, 15, Finished, Available, Finished)

In [15]:
if debug:
    display(products_bad_df)

StatementMeta(, b27fd58b-8e2b-44af-99a1-e8f65a9073b8, 16, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 7fdf2bf4-c0de-463d-9ba1-3dfc1b595046)

In [16]:
products_bad_keys = [row["Product_Code"] for row in products_bad_df.select("Product_Code").distinct().collect()]

StatementMeta(, b27fd58b-8e2b-44af-99a1-e8f65a9073b8, 17, Finished, Available, Finished)

In [17]:
if debug:
    print("Level 1 Bad: " , products_bad_keys)

StatementMeta(, b27fd58b-8e2b-44af-99a1-e8f65a9073b8, 18, Finished, Available, Finished)

Level 1 Bad:  []


In [18]:
if debug:
    display(products_bad_df)
    display(products_df)

StatementMeta(, b27fd58b-8e2b-44af-99a1-e8f65a9073b8, 19, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, f91b873d-6adc-40e1-8d35-b3e7d021b2ed)

SynapseWidget(Synapse.DataFrame, 3e8ec99b-7cb0-4052-bb8d-b4221e296759)

# Init Good File Name & Date Logic

In [19]:
file_path_folder = "/lakehouse/default/Files/Output/"
file_extention = '.dat'

# file date time - stamp tomorrow's date if after 6.15pm --Nick: had to knock it back 1 hour to account for timezone difference; working
now = datetime.now()
cutoff_time = now.replace(hour=17, minute=15, second=0, microsecond=0)

if now > cutoff_time:
    tomorrow = now + timedelta(days=1)
    #file_datetime = tomorrow.strftime('%Y-%m-%d')
    file_datetime = tomorrow.strftime('%Y-%m-%d-%H')
else:
    #file_datetime = now.strftime('%Y-%m-%d')
    file_datetime = now.strftime('%Y-%m-%d-%H')


file_name = "products" + "_" + file_datetime + file_extention
file_path = file_path_folder + file_name

StatementMeta(, b27fd58b-8e2b-44af-99a1-e8f65a9073b8, 20, Finished, Available, Finished)

In [20]:
if debug:
    print("Now: " , now)
    print("Cutoff: " , cutoff_time)
    print("File Date: " , file_datetime)
    print("File Folder Path: " , file_path_folder)
    print("Good File Name: " , file_name)
    print("Good File Name: " , file_path)

StatementMeta(, b27fd58b-8e2b-44af-99a1-e8f65a9073b8, 33, Finished, Available, Finished)

Now:  2025-05-12 08:57:49.556238
Cutoff:  2025-05-12 17:15:00
File Date:  2025-05-12-08
File Folder Path:  /lakehouse/default/Files/Output/
Good File Name:  products_2025-05-12-08.dat
Good File Name:  /lakehouse/default/Files/Output/products_2025-05-12-08.dat


# Remove Rows Already Sent

In [ ]:
if incremental_run:
    
    # REMOVE ROWS FROM CURRENT RUN THAT HAVE ALREADY BEEN SENT (EXIST IN RECORD TRACKING)

    lakehouse_table_name = "bondedwarehouserecordtracking_products"
    container_column = "Product_Code"

    try:
        # Loads already sent Containers from record tracking
        sentrecords_df = spark.read.table(lakehouse_table_name).select(container_column).distinct()

        # Filter each input dataframe to EXCLUDE already sent
        products_df = products_df.join(sentrecords_df, products_df["Product_Code"] == sentrecords_df["Product_Code"], "left_anti")

    except Exception as e:
        print(f"An error occurred: {e}")

StatementMeta(, b27fd58b-8e2b-44af-99a1-e8f65a9073b8, 34, Finished, Available, Finished)

# Export Good

In [21]:
save_dataframe_to_csv(products_df, file_path, show_header=False)

StatementMeta(, b27fd58b-8e2b-44af-99a1-e8f65a9073b8, 35, Finished, Available, Finished)

Add pipes: True
Show headers: False


# Init Bad File Name

In [22]:
error_file = "products_errors_" + file_datetime + file_extention
error_file_path = file_path_folder + error_file

if debug:
    print("file: ", error_file, " path: " , error_file_path)

StatementMeta(, b27fd58b-8e2b-44af-99a1-e8f65a9073b8, 36, Finished, Available, Finished)

file:  products_errors_2025-05-12-08.dat  path:  /lakehouse/default/Files/Output/products_errors_2025-05-12-08.dat


# Export Bad

In [23]:
save_dataframe_to_csv(products_bad_df, error_file_path, show_header = True)

StatementMeta(, b27fd58b-8e2b-44af-99a1-e8f65a9073b8, 37, Finished, Available, Finished)

Add pipes: True
Show headers: True


In [24]:
if products_df.take(1):

    ready_to_copy = True

else:

    ready_to_copy = False

if debug:
    print(ready_to_copy)

StatementMeta(, b27fd58b-8e2b-44af-99a1-e8f65a9073b8, 38, Finished, Available, Finished)

False


# Init Record Tracking

In [25]:
if incremental_run:

    # Save the final_df to different tables based on the exportfile variable value

    def record_tracking_df_to_table(dataframe, table_name, file_name):
        """Saves distinct records to a table, checking for duplicates and enabling column mapping."""
        table_name_lower = table_name.lower()

        # Check if table exists
        table_exists = True
        try:
            spark.read.table(table_name_lower)
            print(f"Table {table_name_lower} exists.")
        except Exception as e:
            print(f"Table {table_name_lower} does not exist.")
            table_exists = False

        # Columns to deduplicate on (excluding metadata)
        dedup_cols = [col for col in dataframe.columns if col not in ["Timestamp", "ExportName", "ExportDate"]]

        if table_exists:
            try:
                dataframe = dataframe.withColumn("ExportName", lit(file_name).cast(StringType())) \
                                    .withColumn("ExportDate", current_timestamp().cast(TimestampType()))

                existing_df = spark.read.table(table_name_lower)

                distinct_existing_df = existing_df.dropDuplicates(subset=dedup_cols)
                initial_existing_count = distinct_existing_df.count()
                print(f"Existing distinct count: {initial_existing_count}")

                distinct_new_df = dataframe.dropDuplicates(subset=dedup_cols)

                combined_distinct_df = distinct_new_df.unionByName(distinct_existing_df) \
                                                    .dropDuplicates(subset=dedup_cols)
                final_distinct_count = combined_distinct_df.count()
                print(f"Final distinct count: {final_distinct_count}")

                rows_added = final_distinct_count - initial_existing_count
                print(f"Added {rows_added} new distinct records to {table_name_lower}.")

                combined_distinct_df.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name_lower)
                print(f"Distinct records saved to {table_name_lower}.")

            except Exception as e:
                print(f"Exception: Saving all records in new table. Exception: {e}")
                dataframe = dataframe.dropDuplicates(subset=dedup_cols)
                dataframe.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name_lower)
                print(f"Distinct records saved to {table_name_lower}.")

        else:
            try:
                print(f"Table doesn't exist. Saving all records in new table.")
                dataframe = dataframe.withColumn("ExportName", lit(file_name).cast(StringType())) \
                                    .withColumn("ExportDate", current_timestamp().cast(TimestampType()))
                dataframe = dataframe.dropDuplicates(subset=dedup_cols)
                dataframe.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name_lower)
                print(f"Distinct records saved to {table_name_lower}.")
            except Exception as e:
                print(f"Error saving data: {e}")

    table_prefix = 'BondedWarehouseRecordTracking_'

    record_tracking_df_to_table(products_df, f"{table_prefix}products", file_name)

StatementMeta(, b27fd58b-8e2b-44af-99a1-e8f65a9073b8, 39, Finished, Available, Finished)

Table bondedwarehouserecordtracking_products exists.


Existing distinct count: 12


Final distinct count: 12
Added 0 new distinct records to bondedwarehouserecordtracking_products.


Distinct records saved to bondedwarehouserecordtracking_products.


# Init Send To Azure Blob Storage

In [26]:
if ready_to_copy == False:
    output_msg = f'Process Complete'

    notebookutils.notebook.exit(output_msg)

StatementMeta(, b27fd58b-8e2b-44af-99a1-e8f65a9073b8, 40, Finished, Available, Finished)

ExitValue: Process Complete

## Copy the file to an ADLS account for loading to the SFTP
Set the source and destination paths

In [27]:
if "DEV" in workspace_name.upper():
    dest_abfss_file_path = "Files/bonded_warehouse_dev/ToBeSent/" + file_name
    
elif "UAT" in workspace_name.upper():
    dest_abfss_file_path = "Files/bonded_warehouse_uat/ToBeSent/" + file_name

else:
    dest_abfss_file_path = "Files/bonded_warehouse/ToBeSent/" + file_name

StatementMeta(, b27fd58b-8e2b-44af-99a1-e8f65a9073b8, -1, Cancelled, , Cancelled)

In [28]:
source_abfss_file_path = 'Files/Output/' + file_name

StatementMeta(, b27fd58b-8e2b-44af-99a1-e8f65a9073b8, -1, Cancelled, , Cancelled)

In [29]:
if transfer_file:
    notebookutils.fs.fastcp(source_abfss_file_path, dest_abfss_file_path)

StatementMeta(, b27fd58b-8e2b-44af-99a1-e8f65a9073b8, -1, Cancelled, , Cancelled)

In [30]:
if ready_to_copy == True:
    output_msg = f'Process Complete'

notebookutils.notebook.exit(output_msg)

StatementMeta(, b27fd58b-8e2b-44af-99a1-e8f65a9073b8, -1, Cancelled, , Cancelled)